# KGGen: extraction and entity clustering

Run cells in order from the repository root. Requires Ollama. Extraction is sequential to avoid competing requests to the local model.

## 1. Configuration and PDF loading

In [ ]:
from pathlib import Path
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from graph_io import export_graph, graph_document_records, visualize_graph

ROOT = Path.cwd()
PDF_PATH = ROOT / "abel-and-his-great-adventure.pdf"
OUTPUT_DIR = ROOT / "outputs"
MODEL = os.getenv("OLLAMA_MODEL", "gpt-oss:20b")
CHUNK_SIZE = 4000
CHUNK_OVERLAP = 200

if not PDF_PATH.is_file():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}. Start Jupyter from the repository root.")
pages = PyPDFLoader(str(PDF_PATH)).load()
if not pages:
    raise ValueError("The PDF contains no pages.")
print(f"Loaded {len(pages)} pages")

## 2. Extract one chunk at a time

Failed chunks are reported and block export until extraction succeeds.

In [ ]:
extraction_complete = False
from kg_gen import KGGen, Graph

kg = KGGen(model=f"ollama/{MODEL}")
story_text = "\n\n".join(page.page_content for page in pages)  # KGGen expects a plain string, not Document objects


# Steers extraction away from literal triple-mining of figurative prose (metaphors, imagery,
# abstract nouns like "soul" / "origin of evil") and toward the things we actually want in the
# graph: characters, places, animals, objects and the events/relations between them.
EXTRACTION_CONTEXT = (
    "This text is a work of literary fiction, not a factual or technical document. "
    "Extract only concrete entities: named people, animals, places, and physical objects, "
    "and the concrete events or relationships between them (e.g. who lives where, who owns "
    "what, who talks to whom, family relations, etc..). Resolve pronouns (he/she/I/you) to the "
    "character's name when it is clear from context. Do NOT extract abstract nouns, emotions, "
    "metaphors, or descriptive imagery (e.g. 'soul', 'origin of evil', 'warm colours') as entities."
)

# Chunking manually and looping sequentially, instead of kg.generate(chunk_size=...): that path
# fires every chunk concurrently via ThreadPoolExecutor, which just queues up against a single
# local Ollama GPU instance and times out. One request at a time matches how Ollama actually
# processes them, so nothing is wasted waiting.
chunk_splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
text_chunks = chunk_splitter.split_text(story_text)
print(f"Split into {len(text_chunks)} chunks")

failed_chunks = []
all_entities = set()
all_relations = set()

for i, chunk in enumerate(text_chunks):
    # gpt-oss:20b occasionally returns an empty response (reasoning-model quirk) and dspy raises
    # instead of returning an empty graph. Skip that one chunk rather than losing everything
    # accumulated so far in the loop.
    try:
        chunk_graph = kg.generate(input_data=chunk, context=EXTRACTION_CONTEXT, cluster=False)
    except Exception as e:
        failed_chunks.append(i + 1)
        print(f"Chunk {i + 1}/{len(text_chunks)} failed, skipping: {e}")
        continue

    all_entities.update(chunk_graph.entities)
    all_relations.update(chunk_graph.relations)
    print(f"Chunk {i + 1}/{len(text_chunks)} done | {len(all_entities)} entities so far")

graph = Graph(
    entities=all_entities,
    relations=all_relations,
    edges={r[1] for r in all_relations},
)

print(f"Entities: {len(graph.entities)}, Relations: {len(graph.relations)}")
if not all_entities:
    raise RuntimeError("Extraction produced no entities; check Ollama and retry.")
if failed_chunks:
    raise RuntimeError(f"Incomplete extraction: failed chunks {failed_chunks}. Resolve the failures and rerun this cell before exporting.")
extraction_complete = True


## 3. Cluster entities

Clustering retries parsing failures. If all retries fail, the unclustered graph is retained and a warning is printed.

In [ ]:
from dspy.utils.exceptions import AdapterParseError

MAX_CLUSTER_ATTEMPTS = 5

# Same gpt-oss:20b empty-response quirk as above, but kg.cluster() runs many LLM calls internally
# (extract/validate/choose-representative per batch) with no retry of its own, so one flaky call
# aborts the whole merge. Kept in its own cell so a failure here doesn't force re-running the
# ~minutes-long extraction loop above — just re-run this cell.
for attempt in range(1, MAX_CLUSTER_ATTEMPTS + 1):
    try:
        clustered_graph = kg.cluster(graph, context=EXTRACTION_CONTEXT)  # merges duplicate/coreferent entities
        graph = clustered_graph
        break
    except AdapterParseError as e:
        print(f"Cluster attempt {attempt}/{MAX_CLUSTER_ATTEMPTS} failed, retrying: {e}")
else:
    print("Clustering failed after all retries; continuing with the unclustered graph.")

print(f"Entities: {len(graph.entities)}, Relations: {len(graph.relations)}")

## 4. Export and visualize

In [ ]:
if not extraction_complete:
    raise RuntimeError("Complete extraction successfully before exporting.")

# Sorting makes JSON ordering stable; extraction itself can still vary between runs.
entities = set(graph.entities)
entities.update(endpoint for s, p, o in graph.relations for endpoint in (s, o))
nodes = [{"id": e, "type": "Entity", "properties": {}} for e in sorted(entities)]
relationships = [
    {"source": s, "source_type": "Entity", "target": o, "target_type": "Entity",
     "type": "_".join(p.upper().split()), "properties": {"predicate": p}}
    for s, p, o in sorted(graph.relations)
]
export_graph(nodes, relationships, OUTPUT_DIR, "kggen")
visualize_graph(nodes, relationships, OUTPUT_DIR / "kggen_graph.html")